In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from src.to_html import buttons, cell, heading, image, paragraph

In [2]:
save_dir = Path("../stats/data")
data_dir = Path("../monthly_processed_data")
files = sorted(data_dir.glob("data-*.parquet"))[-2:]

df_list = []
for file in files:
    df_list.append(pd.read_parquet(file)[["delay_in_min", "time", "is_canceled", "train_type", "train_line_ride_id"]])
df = pd.concat(df_list, ignore_index=True)

# Only look at stops that are not canceled
df = df[~df["is_canceled"]]

# Create a 'time_minutes' column that uses time since 2024-01-01 in minutes
df["time_minutes"] = (df["time"] - pd.Timestamp("2024-01-01")).dt.total_seconds() / 60

print(f"Loaded {len(df):,} records from {len(files)} files")
print(f"Files: {[f.name for f in files]}")

Loaded 28,230,974 records from 2 files
Files: ['data-2025-11.parquet', 'data-2025-12.parquet']


In [3]:
def calculate_delay_progression(data, max_time_since_start):
    """Calculate average delay progression over time since train start."""
    # Group by train_line_ride_id and aggregate time and delay
    grouped = data.groupby("train_line_ride_id").agg({"time_minutes": list, "delay_in_min": list})

    # Calculate time since start and average delay
    all_times = []
    all_delays = []

    for _, row in grouped.iterrows():
        times = np.array(row["time_minutes"])
        delays = row["delay_in_min"]

        all_times.extend(times - np.min(times))  # add times deltas (minutes since first stop)
        all_delays.extend(delays)

    # Create a DataFrame with the results
    result_df = pd.DataFrame({"time_since_start": all_times, "delay_in_min": all_delays})

    # Filter by max_time_since_start
    result_df = result_df[result_df["time_since_start"] <= max_time_since_start]

    # Group by time_since_start and calculate mean delay and count
    avg_delay = result_df.groupby("time_since_start").agg({"delay_in_min": ["mean", "count"]}).reset_index()
    avg_delay.columns = ["time_since_start", "mean_delay", "count"]

    return avg_delay


def plot_delay_progression(avg_delay, filename):
    """Plot delay progression with weighted regression."""
    plt.figure(figsize=(12, 6))

    # Scatter plot with point size proportional to count
    plt.scatter(
        avg_delay["time_since_start"],
        avg_delay["mean_delay"],
        alpha=0.99,
        s=avg_delay["count"] / avg_delay["count"].max() * 1000,
    )

    # Weighted regression
    weights = avg_delay["count"]
    x = avg_delay["time_since_start"]
    y = avg_delay["mean_delay"]

    # Perform weighted least squares regression
    wx = np.average(x, weights=weights)
    wy = np.average(y, weights=weights)
    wx2 = np.average(x**2, weights=weights)
    wxy = np.average(x * y, weights=weights)

    slope = (wxy - wx * wy) / (wx2 - wx**2)
    intercept = wy - slope * wx

    line = slope * x + intercept

    plt.plot(x, line, color="r", label=f"Startverspätung: {intercept:.0f} min, Steigung: {slope * 60:.1f} min/h")

    plt.xlabel("Zeit seit Zugstart (Minuten)")
    plt.ylabel("Durchschnittliche Verspätung (Minuten)")
    plt.title("Durchschnittlicher Verspätungsverlauf")
    plt.legend(loc="upper left")
    plt.grid(True)
    plt.ylim(0, 40)

    max_time = max(avg_delay["time_since_start"])
    xticks = np.arange(0, max_time + 60, 60)
    plt.xticks(xticks)

    plt.savefig(save_dir / filename, dpi=150, bbox_inches="tight")
    plt.close()

In [4]:
# Calculate and plot for all train types
train_type_configs = [
    ("Alle Züge", None, 300),
    ("ICE", "ICE", 480),
    ("IC", "IC", 420),
    ("RB", "RB", 180),
    ("RE", "RE", 180),
    ("S", "S", 180),
]

train_type_contents = {}
for label, train_type, max_time in train_type_configs:
    if train_type is None:
        df_filtered = df
        filename = "verspaetungsverlauf_alle.png"
    else:
        df_filtered = df[df["train_type"] == train_type]
        filename = f"verspaetungsverlauf_{train_type.lower()}.png"

    avg_delay = calculate_delay_progression(df_filtered, max_time_since_start=max_time)
    plot_delay_progression(avg_delay, filename)

    train_type_contents[label] = image(f"../stats/data/{filename}", f"Verspätungsverlauf {label}")

content = cell(
    heading("Wie verändert sich die durchschnittliche Verspätung im Verlauf einer Zugfahrt?", level=1)
    + paragraph(
        "Der Graph zeigt den durchschnittlichen Verspätungsverlauf der Züge über die Zeit. "
        "Die x-Achse stellt die Zeit seit Zugstart in Minuten dar, während die y-Achse die durchschnittliche Verspätung in Minuten anzeigt."
    )
    + paragraph(
        "Jeder Punkt im Diagramm repräsentiert die durchschnittliche Verspätung zu einem bestimmten Zeitpunkt nach der geplanten Abfahrtszeit. "
        "Die Größe der Punkte ist proportional zur Anzahl der Datenpunkte, die in diesen Durchschnitt einfließen. "
        "Größere Punkte bedeuten also, dass mehr Daten für diesen Zeitpunkt verfügbar waren."
    )
    + paragraph(
        "Die rote Linie zeigt den gewichteten Trend der Verspätungen. "
        "Die Steigung dieser Linie gibt an, wie sich die Verspätung im Durchschnitt pro Stunde entwickelt. "
        "Das heißt wenn ich später in einer Zugstrecke in einen Zug einsteige ist die Wahrscheinlichkeit höher, dass der Zug mehr Verspätung hat."
    )
    + buttons(train_type_contents)
)
display(HTML(content))

In [5]:
start_date = files[0].stem.replace("data-", "")
end_date = files[-1].stem.replace("data-", "")
content = cell(
    f'<p style="font-size: 0.8em; text-align: center;">Quelle: <a href="https://github.com/piebro/deutsche-bahn-data/blob/main/notebooks/verspaetungsverlauf.ipynb">Berechnet</a> auf Basis von <a href="https://github.com/piebro/deutsche-bahn-data">gesammelten Daten</a> von der Deutschen Bahn vom {start_date} bis {end_date}.</p>'
)
display(HTML(content))